# Baseline Model
### Evidence Retrieval
- Use Doc2Vec to encode test claims and all evidences
- Compute cosine similarity between claims and each evidence
- Select top 5 evidences that have the highest similarity score for each claim

### Claim Classification
- Use Random Forest to predict claim label

In [2]:
import json
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models.doc2vec import Doc2Vec
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [3]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Clare\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
train_claims_data = load_data('../data/train-claims.json')
evidence_data = load_data('../data/evidence.json')
dev_claims_data = load_data('../data/dev-claims.json')
evidence_map = load_data('../data/curated/preprocessed_evidence_map.json')
test_claims_data = load_data('../data/test-claims-unlabelled.json')

In [6]:
evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])
evidence_df

,id,evidence
0,evidence-0,john bennet law english entrepreneur agricultu...
1,evidence-1,lindberg began profession career age eventu mo...
2,evidence-2,boston ladi cambridg vampir weekend
3,evidence-3,gerald franci goyer born octob profession ice ...
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...
...,...,...
1208822,evidence-1208822,also properti contribut garag apart
1208823,evidence-1208823,class fn org fyrd volda
1208824,evidence-1208824,dragon storm game game collect card game
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...


In [7]:
data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)

train_claims_df['evidence_texts'] = train_claims_df['evidence'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

train_claims_df

,claim,evidence,label,evidence_texts
0,Not only is there no scientific evidence that ...,"[evidence-442946, evidence-1194317, evidence-1...",DISPUTED,[high concentr time atmospher concentr greater...
1,El Niño drove record highs in global temperatu...,"[evidence-338219, evidence-1127398]",REFUTES,[climat chang due natur forc human activ subst...
2,"In 1946, PDO switched to a cool phase.","[evidence-530063, evidence-984887]",SUPPORTS,[evid revers prevail polar mean chang cool sur...
3,Weather Channel co-founder John Coleman provid...,"[evidence-1177431, evidence-782448, evidence-5...",DISPUTED,[convinc scientif evid human releas carbon dio...
4,"""January 2008 capped a 12 month period of glob...","[evidence-1010750, evidence-91661, evidence-72...",NOT_ENOUGH_INFO,"[averag temperatur, iranian persian calendar c..."
...,...,...,...,...
1223,Climate scientists say that aspects of the cas...,"[evidence-1055682, evidence-1047356, evidence-...",SUPPORTS,[fact climat chang made hurrican harvey deadli...
1224,"In its 5th assessment report in 2013, the IPCC...",[evidence-916755],SUPPORTS,[scientif consensu updat state ipcc fifth asse...
1225,"Since the mid 1970s, global temperatures have ...","[evidence-403673, evidence-889933, evidence-11...",NOT_ENOUGH_INFO,"[global warm, multipl independ produc instrume..."
1226,But abnormal temperature spikes in February an...,"[evidence-97375, evidence-562427, evidence-521...",NOT_ENOUGH_INFO,[lower air temperatur record may influenc grou...


In [8]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)

dev_claims_df['evidence_texts'] = dev_claims_df['evidence'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

dev_claims_df

,claim,evidence,label,evidence_texts
0,[South Australia] has the most expensive elect...,"[evidence-67732, evidence-572512]",SUPPORTS,[citat need south australia highest retail pri...
1,when 3 per cent of total annual global emissio...,"[evidence-996421, evidence-1080858, evidence-2...",NOT_ENOUGH_INFO,[unep green economi report state agricultur op...
2,This means that the world is now 1C warmer tha...,"[evidence-889933, evidence-694262]",SUPPORTS,[multipl independ produc instrument dataset co...
3,"“As it happens, Zika may also be a good model ...","[evidence-422399, evidence-702226, evidence-28...",NOT_ENOUGH_INFO,[genet disord result deleteri mutat due sponta...
4,Greenland has only lost a tiny fraction of its...,"[evidence-52981, evidence-264761, evidence-947...",REFUTES,[iceberg calv happen averag greenland lost gt ...
...,...,...,...,...
149,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-409365, evidence-127519, evidence-85...",REFUTES,[state articl convent requir greenhous ga ghg ...
150,"after a natural orbitally driven warming, atmo...","[evidence-368192, evidence-261690, evidence-20...",NOT_ENOUGH_INFO,[increas atmospher concentr co greenhous gase ...
151,Many of the world’s coral reefs are already ba...,"[evidence-1124018, evidence-995813, evidence-1...",NOT_ENOUGH_INFO,[tropic water contain nutrient yet coral reef ...
152,A recent study led by Lawrence Livermore Natio...,[evidence-660755],REFUTES,[studi david douglass cowork conclud commonli ...


In [10]:
model= Doc2Vec.load("../d2v.model")

In [14]:
# evidence_df["vector"] = ""
# for i in range(evidence_df.shape[0]):
#     inferred_vector = model.infer_vector(evidence_df["evidence"][i].split())
#     evidence_df["vector"][i] = inferred_vector
# evidence_df.to_csv('../data/curated/evidence_vector.csv', index=False)

evidence_df = pd.read_csv('../data/curated/evidence_vector.csv', converters={
    'vector': lambda x: np.array(x.strip("[]").split(), dtype='float')})
evidence_df

,id,evidence,vector
0,evidence-0,john bennet law english entrepreneur agricultu...,"[0.02834939, -0.07332291, -0.0519409, 0.090428..."
1,evidence-1,lindberg began profession career age eventu mo...,"[0.05698019, 0.1287728, -0.37075773, -0.298026..."
2,evidence-2,boston ladi cambridg vampir weekend,"[0.03032708, -0.22458611, 0.14641732, 0.343893..."
3,evidence-3,gerald franci goyer born octob profession ice ...,"[0.12884548, -0.10786802, -0.33527467, -0.4029..."
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...,"[0.15580554, -0.06295265, -0.07655199, -0.0584..."
...,...,...,...
1208822,evidence-1208822,also properti contribut garag apart,"[0.1791487, 0.1119303, -0.107538, -0.22217059,..."
1208823,evidence-1208823,class fn org fyrd volda,"[0.02914719, 0.15489092, -0.11366356, -0.11779..."
1208824,evidence-1208824,dragon storm game game collect card game,"[0.1677031, 0.12748027, -0.3443975, -0.3463705..."
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...,"[0.46210417, 0.09757853, -0.41667572, -0.20164..."


In [15]:
data_for_dataframe = []
for claim_id, claim_details in test_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'claim_text_raw': claim_details['claim_text']
        })
    
# Create DataFrame
test_claims_df = pd.DataFrame(data_for_dataframe)
test_claims_df 

,claim_id,claim,claim_text_raw
0,claim-2967,contribut wast heat global climat,The contribution of waste heat to the global c...
1,claim-979,warm weather worsen recent drought includ drie...,“Warm weather worsened the most recent five-ye...
2,claim-1609,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...
3,claim-1020,global reef crisi necessarili mean extinct cor...,“The global reef crisis does not necessarily m...
4,claim-2599,small amount activ substanc caus larg effect,Small amounts of very active substances can ca...
...,...,...,...
148,claim-293,measur equip get old need replac often requir,When the measuring equipment gets old and need...
149,claim-910,cement iron steel petroleum refin industri cou...,"The cement, iron and steel, and petroleum refi..."
150,claim-2815,new studi surfac warm solar cycl found time hi...,A new peer-reviewed study on Surface Warming a...
151,claim-1652,strong co2 effect observ mani differ measur,The strong CO2 effect has been observed by man...


In [17]:
test_claims_df["vector"] = ""
for i in range(test_claims_df.shape[0]):
    inferred_vector = model.infer_vector(test_claims_df["claim"][i].split())
    test_claims_df["vector"][i] = inferred_vector
test_claims_df.to_csv('../data/curated/test_claim_vector.csv', index=False)
test_claims_df 

,claim_id,claim,claim_text_raw,vector
0,claim-2967,contribut wast heat global climat,The contribution of waste heat to the global c...,"[0.07331854, 0.05240894, -0.12168409, -0.16622..."
1,claim-979,warm weather worsen recent drought includ drie...,“Warm weather worsened the most recent five-ye...,"[-0.03228369, 0.3208165, -0.35173163, 0.060934..."
2,claim-1609,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...,"[0.054185323, 0.106044814, -0.031156693, -0.07..."
3,claim-1020,global reef crisi necessarili mean extinct cor...,“The global reef crisis does not necessarily m...,"[0.1440339, 0.053078555, -0.20366696, -0.17837..."
4,claim-2599,small amount activ substanc caus larg effect,Small amounts of very active substances can ca...,"[0.03418594, 0.13198684, -0.13199791, -0.24544..."
...,...,...,...,...
148,claim-293,measur equip get old need replac often requir,When the measuring equipment gets old and need...,"[0.05995972, 0.16992326, -0.04714366, -0.24887..."
149,claim-910,cement iron steel petroleum refin industri cou...,"The cement, iron and steel, and petroleum refi...","[0.10882567, 0.25015888, -0.2738379, -0.507496..."
150,claim-2815,new studi surfac warm solar cycl found time hi...,A new peer-reviewed study on Surface Warming a...,"[0.35375205, 0.2388189, 0.0014900521, -0.02331..."
151,claim-1652,strong co2 effect observ mani differ measur,The strong CO2 effect has been observed by man...,"[0.009911904, 0.20787366, -0.09120769, -0.2891..."


In [18]:
X = np.array(test_claims_df['vector'].values.tolist())
y = np.array(evidence_df['vector'].values.tolist())
sim = cosine_similarity(X, y)
print(sim.shape)
    

(153, 1208827)


In [19]:
# get top 5 evidence with highest similarity score with the claim
data = np.zeros((sim.shape[0], 5))
for i in range(sim.shape[0]):
	data[i] = np.argpartition(sim[i], -5)[-5:]
data = data.astype(np.int32)

test_claims_df['top5_evidence_id'] = data.tolist()
test_claims_df = test_claims_df[["claim_id", "claim_text_raw", "top5_evidence_id"]]

# get texts of top 5 evidence
test_claims_df['evidence_texts'] = test_claims_df['top5_evidence_id'].apply(
    lambda x: [evidence_map["evidence-" + str(evidence_id)] for evidence_id in x]
)

test_claims_df.to_csv("../data/curated/test_evidence_retrieval.csv", index=False)
test_claims_df

C:\Users\Clare\AppData\Local\Temp\ipykernel_9304\3065721464.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_claims_df['evidence_texts'] = test_claims_df['top5_evidence_id'].apply(


,claim_id,claim_text_raw,top5_evidence_id,evidence_texts
0,claim-2967,The contribution of waste heat to the global c...,"[1065948, 1042695, 573930, 833667, 533238]","[also call combin heat power district heat, cy..."
1,claim-979,“Warm weather worsened the most recent five-ye...,"[496086, 733944, 947706, 333873, 410789]","[accord fao averag annual rainfal millimet mm,..."
2,claim-1609,Greenland has only lost a tiny fraction of its...,"[1055742, 274316, 7081, 1175803, 197579]","[δd ice, andrea vaturi born septemb italian fo..."
3,claim-1020,“The global reef crisis does not necessarily m...,"[265282, 914949, 305176, 455747, 451684]",[declar express concern unfound undu concern e...
4,claim-2599,Small amounts of very active substances can ca...,"[468537, 398105, 532261, 1136925, 602236]",[tom carter born januari american profession g...
...,...,...,...,...
148,claim-293,When the measuring equipment gets old and need...,"[62653, 472007, 866036, 781111, 522857]","[hous demolish later date without announc, int..."
149,claim-910,"The cement, iron and steel, and petroleum refi...","[33053, 916128, 792538, 653202, 989902]",[więcki wjencki wenzken villag administr distr...
150,claim-2815,A new peer-reviewed study on Surface Warming a...,"[283869, 575139, 251169, 485917, 968454]",[transient respons lower equilibrium sensit de...
151,claim-1652,The strong CO2 effect has been observed by man...,"[933850, 1152725, 44072, 1144764, 842337]",[embargo caus oil crisi mani effect global pol...


### Claim Classification

In [ ]:
# dev_claims_df = pd.read_csv('data/curated/dev_evidence_retrieval.csv', converters={
#     'top5_evidence_id': lambda x: np.array(x.strip("[]").split(','), dtype='int')})
# dev_claims_df

In [23]:
# label_mapping = {
#     "SUPPORTS": 1,
#     "REFUTES": 2,
#     "DISPUTED": 3,
#     "NOT_ENOUGH_INFO": 0
# }
data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    # if claim_label == "DISPUTED":
    #         continue
    for eid in eids:
        evidence_text = evidence_map[eid]
        
        data_for_dataframe.append({
                'claim_id': claim_id,
                'claim': preprocess_text(claim_text),
                'evidence': evidence_text,
                # 'label': label_mapping[claim_label]
                'label': claim_label
            })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)

# train_claims_df['evidence_texts'] = train_claims_df['evidence'].apply(
#     lambda x: ' '.join([evidence_map[evidence_id] for evidence_id in x])
# )

train_claims_df

,claim_id,claim,evidence,label
0,claim-1937,scientif evid co2 pollut higher co2 concentr a...,high concentr time atmospher concentr greater ...,DISPUTED
1,claim-1937,scientif evid co2 pollut higher co2 concentr a...,plant grow much percent faster concentr ppm co...,DISPUTED
2,claim-1937,scientif evid co2 pollut higher co2 concentr a...,higher carbon dioxid concentr favour affect pl...,DISPUTED
3,claim-126,el niño drove record high global temperatur su...,climat chang due natur forc human activ substa...,REFUTES
4,claim-126,el niño drove record high global temperatur su...,acceler due mostli global warm drive thermal e...,REFUTES
...,...,...,...,...
4117,claim-502,abnorm temperatur spike februari earlier month...,coastlin see significantli mild temperatur com...,NOT_ENOUGH_INFO
4118,claim-3093,send oscil microwav antenna insid vacuum elect...,dielectr heat also known electron heat radio f...,SUPPORTS
4119,claim-3093,send oscil microwav antenna insid vacuum elect...,exampl absorpt emiss radio wave antenna absorp...,SUPPORTS
4120,claim-3093,send oscil microwav antenna insid vacuum elect...,water fat substanc food absorb energi microwav...,SUPPORTS


In [25]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    # if claim_label == "DISPUTED":
    #         continue
    for eid in eids:
        evidence_text = evidence_map[eid]
        
        data_for_dataframe.append({
                'claim_id': claim_id,
                'claim': preprocess_text(claim_text),
                'evidence': evidence_text,
                # 'label': label_mapping[claim_label]
                'label': claim_label
            })

# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df

,claim_id,claim,evidence,label
0,claim-752,south australia expens electr world,citat need south australia highest retail pric...,SUPPORTS
1,claim-752,south australia expens electr world,south australia highest power price world,SUPPORTS
2,claim-375,3 per cent total annual global emiss carbon di...,unep green economi report state agricultur ope...,NOT_ENOUGH_INFO
3,claim-375,3 per cent total annual global emiss carbon di...,market share potenti clean electr heat pump co...,NOT_ENOUGH_INFO
4,claim-375,3 per cent total annual global emiss carbon di...,modern era emiss atmospher volcano approxim bi...,NOT_ENOUGH_INFO
...,...,...,...,...
486,claim-1426,mani world coral reef alreadi barren state con...,aquacultur show promis potenti effect tool res...,NOT_ENOUGH_INFO
487,claim-1426,mani world coral reef alreadi barren state con...,rapidli result transit barren landscap rel spe...,NOT_ENOUGH_INFO
488,claim-698,recent studi led lawrenc livermor nation labor...,studi david douglass cowork conclud commonli u...,REFUTES
489,claim-1021,coral may save mani creatur attempt move towar...,poleward migrat coral speci refer phenomenon b...,SUPPORTS


In [28]:
data_for_dataframe = []
for i, row in test_claims_df.iterrows():
    for evidence in row['evidence_texts']:
        data_for_dataframe.append({
                'claim_id': row['claim_id'],
                'claim': row['claim_text_raw'],
                'evidence': evidence,
            })

test_df_for_classification = pd.DataFrame(data_for_dataframe)
test_df_for_classification

,claim_id,claim,evidence
0,claim-2967,The contribution of waste heat to the global c...,also call combin heat power district heat
1,claim-2967,The contribution of waste heat to the global c...,cyclon wast heat engin whe uniflow steam engin
2,claim-2967,The contribution of waste heat to the global c...,kelottijärvi villag lake municip enontekiö lap...
3,claim-2967,The contribution of waste heat to the global c...,dao maniwo papuan languag spoken paniai lake r...
4,claim-2967,The contribution of waste heat to the global c...,wast commonli use manufactur industri often co...
...,...,...,...
760,claim-1212,"(In technical lingo, the so-called social cost...",armstrong invest manag llp aim global invest m...
761,claim-1212,"(In technical lingo, the so-called social cost...",district lincolnshir lincoln north kesteven so...
762,claim-1212,"(In technical lingo, the so-called social cost...",howev commit often larg aspir much carbon pric...
763,claim-1212,"(In technical lingo, the so-called social cost...",individu lot often


In [29]:
X_train = train_claims_df['claim'] + train_claims_df['evidence']
y_train = train_claims_df['label']

X_dev = dev_claims_df['claim'] + dev_claims_df['evidence']
y_dev = dev_claims_df['label']

X_test = test_df_for_classification['claim'] + test_df_for_classification['evidence']

count_vectorizer = CountVectorizer()
X_train_count = count_vectorizer.fit_transform(X_train)
X_dev_count = count_vectorizer.transform(X_dev)
X_test_count = count_vectorizer.transform(X_test)

In [20]:
# combine claim text and evidence texts
# X_train = train_claims_df['claim'] + train_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
# y_train = train_claims_df['label']

# X_dev = dev_claims_df['claim'] + dev_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
# y_dev = dev_claims_df['label']

# X_test = test_claims_df['claim_text_raw'] + test_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))

# count_vectorizer = CountVectorizer()
# X_train_count = count_vectorizer.fit_transform(X_train)
# X_dev_count = count_vectorizer.transform(X_dev)
# X_test_count = count_vectorizer.transform(X_test)

In [30]:
# Hyperparameters
n_estimators_values = [50, 100, 200]
max_depth_values = [None, 10, 20]

accuracy_scores_rf = []
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        rf_classifier = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        rf_classifier.fit(X_train_count, y_train)
        y_pred_rf = rf_classifier.predict(X_dev_count)
        
        accuracy_rf = accuracy_score(y_dev, y_pred_rf)
        accuracy_scores_rf.append(((n_estimators, max_depth), accuracy_rf))
        print(f"n_estimators = {n_estimators}, max_depth = {max_depth}: Accuracy = {accuracy_rf}")

print("Accuracy scores for Random Forest:")
for params, accuracy in accuracy_scores_rf:
    print(f"Parameters: {params}, Accuracy: {accuracy}")

n_estimators = 50, max_depth = None: Accuracy = 0.4989816700610998
n_estimators = 50, max_depth = 10: Accuracy = 0.4378818737270876
n_estimators = 50, max_depth = 20: Accuracy = 0.4663951120162933
n_estimators = 100, max_depth = None: Accuracy = 0.5030549898167006
n_estimators = 100, max_depth = 10: Accuracy = 0.4480651731160896
n_estimators = 100, max_depth = 20: Accuracy = 0.4745417515274949
n_estimators = 200, max_depth = None: Accuracy = 0.505091649694501
n_estimators = 200, max_depth = 10: Accuracy = 0.45213849287169044
n_estimators = 200, max_depth = 20: Accuracy = 0.46435845213849286
Accuracy scores for Random Forest:
Parameters: (50, None), Accuracy: 0.4989816700610998
Parameters: (50, 10), Accuracy: 0.4378818737270876
Parameters: (50, 20), Accuracy: 0.4663951120162933
Parameters: (100, None), Accuracy: 0.5030549898167006
Parameters: (100, 10), Accuracy: 0.4480651731160896
Parameters: (100, 20), Accuracy: 0.4745417515274949
Parameters: (200, None), Accuracy: 0.505091649694501
P

In [ ]:
# Apply to Test Set
rf_classifier = RandomForestClassifier(n_estimators=50, max_depth=None, random_state=42)
rf_classifier.fit(X_train_count, y_train)
y_pred = rf_classifier.predict(X_test_count)
test_claims_df["label"] = y_pred
test_claims_df['evidences'] = test_claims_df['top5_evidence_id'].apply(
    lambda x: ["evidence-" + str(evidence_id) for evidence_id in x]
)
test_claims_df.drop(columns=['evidence_texts', 'top5_evidence_id'], inplace=True)
test_claims_df.rename(columns={"claim_text_raw": "claim_text", "label": "claim_label"}, inplace=True)
test_claims_df.set_index('claim_id', inplace=True)
test_claims_df

,claim_text,claim_label,evidences
claim_id,,,
claim-2967,The contribution of waste heat to the global c...,SUPPORTS,"[evidence-1071666, evidence-103761, evidence-2..."
claim-979,“Warm weather worsened the most recent five-ye...,SUPPORTS,"[evidence-733944, evidence-947706, evidence-57..."
claim-1609,Greenland has only lost a tiny fraction of its...,SUPPORTS,"[evidence-446869, evidence-1041564, evidence-5..."
claim-1020,“The global reef crisis does not necessarily m...,SUPPORTS,"[evidence-372770, evidence-677084, evidence-91..."
claim-2599,Small amounts of very active substances can ca...,SUPPORTS,"[evidence-816943, evidence-542087, evidence-32..."
...,...,...,...
claim-293,When the measuring equipment gets old and need...,SUPPORTS,"[evidence-1205183, evidence-1016047, evidence-..."
claim-910,"The cement, iron and steel, and petroleum refi...",SUPPORTS,"[evidence-1041788, evidence-777489, evidence-2..."
claim-2815,A new peer-reviewed study on Surface Warming a...,SUPPORTS,"[evidence-968454, evidence-558597, evidence-96..."


In [ ]:
# convert to json file
from json import loads
result = test_claims_df.to_json(orient="index")
with open('data/curated/test-output.json', 'w') as f:
    f.write(result)